<a href="https://colab.research.google.com/github/5821708/5821708-AnalyticsEngineering/blob/main/practice_bike_hire_MODEL_ANSWER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# Practice session
## Bike-hire trips

**MODEL ANSWER**

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [ ]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [ ]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [ ]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

**Grain, from looking at each table**
- `trips`: one row per trip (`trip_id`). `bikes`: one row per bike. `stations`: one row per station.
- My final table: **one row per bike type and start-station zone**, so I need `trips` joined to `bikes` (on `bike_id`) and `stations` (on `start_station_id`). Both lookups should be many trips to one bike or station, so the row count of `trips` should not change.

**Plan before profiling:** I expect missing durations, odd labels and maybe duplicates. I will check, not assume.

First, I will check for nulls / missing data. Should be a list of fields and counts of N/As.

In [ ]:
# Profile: nulls, duplicates, and the distinct values of the label columns
print(tables["trips"].isna().sum(), "\n")

trip_id              0
bike_id              0
start_station_id     0
end_station_id       0
start_time           0
duration_min        30
user_type            0
dtype: int64 



30 missing records in duration_min. Might mean these weren't finished when the data was collected? Probably should check with a subject matter expert.

Now we can check for duplicated rows. Expecting to see a count of duplicates - hopefully zero.

In [ ]:
print("duplicate trip rows:", tables["trips"].duplicated().sum())

duplicate trip rows: 12


12 duplicates. Lets count how many trip_id's are unique (they should all be - expectation as above):

In [ ]:
print("distinct trip_id:", tables["trips"]["trip_id"].nunique(), "of", len(tables["trips"]), "\n")

distinct trip_id: 600 of 612 



This shows the same result - we duplicates.

Now let's count the number of bike and user types. I expect to see a list of types and their counts. Hopefully just two.

In [ ]:
print(tables["bikes"]["bike_type"].value_counts(dropna=False), "\n")
print(tables["trips"]["user_type"].value_counts(dropna=False), "\n")

bike_type
Standard     36
Electric      9
STANDARD      8
standard      3
Electric      2
electric      1
E-bike        1
Name: count, dtype: int64 

user_type
Member     331
Casual     190
member      35
MEMBER      28
CASUAL      11
Casual       9
casual       8
Name: count, dtype: int64 



Some text is in capitals, some in lower, "electric" and "E-bike" are probably the same ... overall a bit of a mess.

Let's  look at the duration values and see if there are any outliers etc. I want to see the summary statistics such as mean, min, max, etc.

In [ ]:
print(tables["trips"]["duration_min"].describe())

count     582.000000
mean       27.891753
std       145.064305
min         4.000000
25%        12.000000
50%        16.000000
75%        22.000000
max      2880.000000
Name: duration_min, dtype: float64


The max value looks very large, as does  the standard deviation (which might be linked).

Let's also look at start times and see if they are all in correct date format. I expect to be able to visually look at some of the results and see if there are problems, and to get a count of how many are separated by "/" and how many by "-":

In [ ]:
# Inspect the unique formats or sample values of start_time
print("Sample start times:")
print(tables["trips"]["start_time"].head(10))
print("\nNumber of values containing '/':", tables["trips"]["start_time"].str.contains("/").sum())
print("Number of values containing '-':", tables["trips"]["start_time"].str.contains("-").sum())

Sample start times:
0    2025-03-01 05:52:00
1    2025-03-01 06:31:00
2    2025-03-01 07:06:00
3    2025-03-01 07:06:00
4    2025-03-01 07:36:00
5    2025-03-01 08:06:00
6       01/03/2025 09:26
7    2025-03-01 10:35:00
8    2025-03-01 12:14:00
9    2025-03-01 13:04:00
Name: start_time, dtype: object

Number of values containing '/': 98
Number of values containing '-': 514


Some are with a "/" and some with a "-". We better do something to fix this.

Last we can look at stations and see if any are missing. I expect to see the fields from the station table and a count of how many are missing (hopefully they are all 0).

In [ ]:
print(tables["stations"].isna().sum())

station_id      0
station_name    0
zone            0
capacity        1
dtype: int64


**What I found**
- Exact duplicate trip rows (same `trip_id` appears twice).
- `bike_type` has inconsistent labels (case, trailing spaces, "E-bike").
- `duration_min` has missing values and a few extreme values (a full day or more).
- One station has no `capacity`, which I do not need for this question, so I will leave it alone.

## 2. Clean and prepare

**Decisions and reasons**
- Drop exact duplicate rows: the same `trip_id` twice is one trip recorded twice.
- Standardise `bike_type` to two values, Standard and Electric.
- Missing duration: exclude those trips from the *average*, but keep a count, because filling them with a guess would change the answer.
- Extreme durations (over 600 minutes): I will exclude them from the average and report how many. They look like bikes not returned, not real trips, but I cannot confirm that.
- Parse `start_time` with two explicit formats, not by guessing.

We'll start with deduplication. If it works we should go to 600 total rows (dropping the 12 duplicates) and there should be duplication left.

In [ ]:
trips = tables["trips"].drop_duplicates().copy()
print("rows before/after dropping duplicates:", len(tables["trips"]), len(trips))
print(trips["trip_id"].is_unique)

rows before/after dropping duplicates: 612 600
True


Gone to 600 rows and True says everything is unique.

Next let's try to fix the bike name. I expect every bike to be named "Standard" or "Electric" if this works.

In [ ]:
bikes = tables["bikes"].copy()
bt = bikes["bike_type"].str.strip().str.lower()
bikes["bike_type"] = bt.map({"standard": "Standard", "electric": "Electric", "e-bike": "Electric"})
print(bikes["bike_type"].value_counts(dropna=False))   # no NaN means every label was mapped

bike_type
Standard    47
Electric    13
Name: count, dtype: int64


Much cleaner! Next is the date formats. My AI gave me this code. I would expect it to show that all dates are in March, if it hasn't processed the "/" ones the same as the "-" ones this will fail.

In [ ]:
# Two date formats: parse each group separately
slash = trips["start_time"].str.contains("/")
trips["start_ts"] = pd.NaT
trips.loc[slash, "start_ts"] = pd.to_datetime(trips.loc[slash, "start_time"], format="%d/%m/%Y %H:%M")
trips.loc[~slash, "start_ts"] = pd.to_datetime(trips.loc[~slash, "start_time"], format="%Y-%m-%d %H:%M:%S")
trips["start_ts"] = pd.to_datetime(trips["start_ts"])
print("all dates in March 2025:", trips["start_ts"].between("2025-03-01", "2025-03-31 23:59:59").all())

all dates in March 2025: True
slash dates with day > 12: 61


Output as expected.

Running out of time so will move on to joing the date.

## 3. Join and aggregate

I expect the row count to stay the same after both joins, because each trip has one bike and one start station.

In [ ]:
con.register("trips_clean", trips)
con.register("bikes_clean", bikes)

joined = q("""
    SELECT t.trip_id, b.bike_type, s.zone, t.duration_ok
    FROM trips_clean AS t
    LEFT JOIN bikes_clean AS b ON t.bike_id = b.bike_id
    LEFT JOIN stations AS s ON t.start_station_id = s.station_id
""")
print("rows in trips:", len(trips), "| rows after joins:", len(joined))

result = q("""
    SELECT bike_type, zone,
           COUNT(*) AS trips,
           COUNT(duration_ok) AS trips_with_valid_duration,
           ROUND(AVG(duration_ok), 1) AS avg_duration_min
    FROM (
        SELECT b.bike_type, s.zone, t.duration_ok
        FROM trips_clean AS t
        LEFT JOIN bikes_clean AS b ON t.bike_id = b.bike_id
        LEFT JOIN stations AS s ON t.start_station_id = s.station_id
    )
    GROUP BY bike_type, zone
    ORDER BY bike_type, zone
""")
result

rows in trips: 600 | rows after joins: 600


,bike_type,zone,trips,trips_with_valid_duration,avg_duration_min
0,Electric,Centre,87,81,14.5
1,Electric,East,12,11,15.7
2,Electric,North,24,23,16.0
3,Electric,South,26,24,13.4
4,Electric,West,25,22,15.7
5,Standard,Centre,227,215,18.9
6,Standard,East,24,20,19.5
7,Standard,North,39,39,20.4
8,Standard,South,60,59,19.5
9,Standard,West,76,73,19.8


## 4. Check

- **Row count:** unchanged by the joins (above), so no fan-out.
- **Totals:** the `trips` column of the result should add up to the cleaned trip count.
- **One number recalculated a different way:** the overall average duration, using pandas on the cleaned data, compared with a trip-weighted average of the table.

In [ ]:
print("sum of trips in result:", result["trips"].sum(), "| cleaned trips:", len(trips))
direct = trips["duration_ok"].mean()
weighted = (result["avg_duration_min"] * result["trips_with_valid_duration"]).sum() / result["trips_with_valid_duration"].sum()
print("overall average (pandas):", round(direct, 2), "| weighted from the table:", round(weighted, 2))

sum of trips in result: 600 | cleaned trips: 600
overall average (pandas): 18.04 | weighted from the table: 18.04


## 5. Prompts and AI notes (an example of what good logging looks like)

*These are illustrative examples written for teaching, not a real log. Use the same shape.*

**Prompt 1:** "Here are the columns of my three tables [pasted]. Write DuckDB SQL for average trip duration by bike type and zone of the start station."

**What I accepted/changed:** the SQL ran, but it joined on the wrong column and did not handle the messy bike labels. I added the cleaning step first and checked the join keys myself.

**Prompt 2:** "Why might the average be wrong if some durations are missing?"

**What I did:** the answer matched my own thinking. I kept the missing and extreme counts visible instead of silently dropping rows.

**A typical AI mistake to watch for:** an answer that fills missing durations with zero or the mean. It runs without error but changes the averages. Catching it needs a check, such as comparing counts before and after.

## 6. What I could not verify

- I assumed durations over 600 minutes are unreturned bikes or trips that did not finish when the data was collected, not real trips. The data does not say.
- I did not look at whether trips ending at a different station matter to the question.
- With more time I would check whether the missing durations cluster on particular bikes or days.